# Clase 18 · R y Visualización de Datos

**Cuaderno de trabajo: todo lo que puedes hacer desde R para visualizar datos**

*Visualización de Datos · Universidad EAN*

> Versión para Google Colab / Jupyter con kernel R. En Colab: **Entorno de ejecución → Cambiar tipo de entorno → R**. Ejecuta las celdas en orden con Shift + Enter. Los bloques que en RStudio no se ejecutan (apps Shiny, ejercicios con `___`) aparecen aquí como texto para que *Ejecutar todo* no se detenga.

In [ ]:
# Tamaño de las figuras en el kernel de R (IRkernel)
options(repr.plot.width = 9, repr.plot.height = 5, repr.plot.res = 110, warn = -1)

> **Cómo usar este cuaderno.** Es un *R Notebook*: cada bloque gris es un *chunk* de código.
> Ponte sobre él y pulsa **Ctrl + Shift + Enter** (o el triángulo verde) para ejecutarlo; el resultado aparece justo debajo.
> Lee el texto, ejecuta el código, **modifícalo** y observa qué cambia. Las secciones marcadas con 🧪 **Tu turno** son ejercicios.
>
> Acompaña la presentación *R y Visualización de Datos: Potencia y Creatividad en el Análisis Estadístico* (Clase 18) y sigue su misma ruta:
> **1)** qué es R → **2)** fundamentos con ggplot2 → **3)** interactividad y dashboards → **4)** flujo completo de análisis.

# Preparación del entorno

## Instalar y cargar paquetes

R trae un sistema gráfico incorporado, pero la mayor parte de la potencia está en los **paquetes** de CRAN (más de 19 000). Este bloque instala **solo los que falten**.

In [ ]:
paquetes <- c("tidyverse", "scales", "patchwork", "ggrepel", "ggridges", "ggcorrplot",
              "viridis", "RColorBrewer", "hexbin", "plotly", "DT", "sf", "svglite")
faltan <- paquetes[!sapply(paquetes, requireNamespace, quietly = TRUE)]
if (length(faltan) > 0) install.packages(faltan, repos = "https://cloud.r-project.org")

# Opcionales (se usan en secciones marcadas como tales; instálalos cuando los necesites):
# install.packages(c("leaflet", "highcharter", "gganimate", "gifski", "shiny", "flexdashboard", "gapminder"))

In [ ]:
library(tidyverse)   # carga ggplot2, dplyr, tidyr, readr, forcats, stringr, lubridate, purrr y tibble
library(scales)      # etiquetas de ejes: porcentajes, dinero, miles...
library(patchwork)   # combinar varios gráficos
library(ggrepel)     # etiquetas que no se superponen
library(ggridges)    # densidades apiladas (ridgelines)
library(ggcorrplot)  # matrices de correlación

# Paleta de la clase (verde EAN + ámbar) para usar en todos los ejemplos
verde <- "#0B6E4F"; verde_oscuro <- "#0A3D2E"; ambar <- "#F2A93B"; gris <- "#5B6B6F"
paleta <- c("#0B6E4F", "#F2A93B", "#3E7CB1", "#B8412D", "#6C4F9A", "#2A9D8F", "#8D6E63")
theme_set(theme_minimal(base_size = 13))   # tema por defecto para todo el cuaderno

## Los datos que usaremos

Todos vienen incluidos en R o en ggplot2, así que no hay que descargar nada:

| Dataset | Contenido | Lo usamos para |
|---|---|---|
| `mpg` | 234 modelos de autos (EPA, 1999-2008): cilindrada, rendimiento, tracción, clase | casi todos los ejemplos |
| `diamonds` | 53 940 diamantes: quilates, precio, corte, claridad | distribuciones y datasets grandes |
| `economics` | Serie mensual de EE. UU. 1967-2015: desempleo, ahorro | series de tiempo |
| `txhousing` | Ventas de vivienda en 46 ciudades de Texas (2000-2015) | el caso práctico completo |
| `mtcars`, `iris`, `airquality` | clásicos de R | correlaciones y ejercicios |

In [ ]:
glimpse(mpg)

In [ ]:
head(mpg, 10)

# Capítulo 1 · Los tres sistemas gráficos de R

R tiene **tres** formas de dibujar. Conocerlas te ayuda a leer código ajeno y a elegir bien:

1. **Gráficos base** (`plot()`, `hist()`, `barplot()`...): vienen con R, son rápidos para *mirar* datos, pero cada ajuste es un comando distinto.
2. **lattice**: paneles condicionados (`xyplot(y ~ x | grupo)`), muy usado en estadística clásica.
3. **ggplot2** (sobre el sistema *grid*): la *gramática de gráficos*. Es el estándar actual y el foco de esta clase.

## Gráficos base: rápidos para explorar

In [ ]:
options(repr.plot.height = 6)
par(mfrow = c(2, 2), mar = c(4, 4, 2.5, 1))                 # cuadrícula 2x2
hist(mpg$hwy, col = verde, border = "white", main = "hist()", xlab = "mpg carretera")
plot(mpg$displ, mpg$hwy, pch = 19, col = adjustcolor(verde, 0.5),
     main = "plot()", xlab = "Cilindrada (L)", ylab = "mpg carretera")
abline(lm(hwy ~ displ, data = mpg), col = ambar, lwd = 2)  # recta de regresión encima
barplot(table(mpg$class), col = ambar, las = 2, main = "barplot()")
boxplot(hwy ~ drv, data = mpg, col = "#CFE8DF", main = "boxplot()")
par(mfrow = c(1, 1))
options(repr.plot.height = 5)

## lattice: paneles por grupo

In [ ]:
lattice::xyplot(hwy ~ displ | drv, data = mpg, layout = c(3, 1),
                main = "lattice::xyplot(hwy ~ displ | drv): un panel por tipo de tracción")

## ¿Por qué ggplot2?

| Gráficos base | ggplot2 |
|---|---|
| Comandos dispersos (`plot`, `lines`, `legend`, `par`...) | **Sintaxis coherente**: todo se construye con `+` |
| Personalizar = recordar decenas de argumentos | **Personalización avanzada** con `scale_*`, `theme()` y `labs()` |
| Leyendas y paneles a mano | Leyendas, facetas y escalas **automáticas** |
| Difícil de reproducir y escalar | **Calidad publicable** y código reutilizable |

# Capítulo 2 · La gramática de gráficos con ggplot2

## Un gráfico se construye por capas

Cada elemento que añades con `+` es una **capa** que se superpone a las anteriores. Ejecuta los cuatro bloques en orden y observa cómo crece el gráfico.

In [ ]:
# Capa 1: datos + estética. Solo el lienzo: ggplot2 ya sabe qué va en cada eje
ggplot(data = mpg, mapping = aes(x = displ, y = hwy))

In [ ]:
# Capa 2: geometría. Cada fila del data frame se convierte en un punto
ggplot(mpg, aes(displ, hwy)) +
  geom_point()

In [ ]:
# Capa 3: estadística. geom_smooth() calcula y dibuja una tendencia
ggplot(mpg, aes(displ, hwy)) +
  geom_point() +
  geom_smooth(method = "loess", formula = y ~ x)

In [ ]:
# Capa 4: escalas, etiquetas y tema. De "exploratorio" a "comunicativo"
ggplot(mpg, aes(displ, hwy, colour = drv)) +
  geom_point(alpha = 0.7, size = 2) +
  geom_smooth(method = "loess", formula = y ~ x, se = FALSE) +
  scale_colour_manual(values = paleta[1:3], labels = c("4x4", "Delantera", "Trasera"), name = "Tracción") +
  labs(title = "A mayor cilindrada, menor rendimiento en carretera",
       subtitle = "234 modelos de autos, 1999-2008",
       x = "Cilindrada del motor (litros)", y = "Rendimiento en carretera (mpg)",
       caption = "Fuente: EPA vía ggplot2::mpg") +
  theme_minimal() +
  theme(legend.position = "top")

Los **siete componentes** de la gramática (Wilkinson, 2005; Wickham, 2010):

| Componente | Pregunta que responde | Función |
|---|---|---|
| **Datos** | ¿Qué tabla? (formato *tidy*) | `ggplot(data = ...)` |
| **Estética** | ¿Qué variable va a qué propiedad visual? | `aes(x, y, colour, size, shape, alpha, fill)` |
| **Geometría** | ¿Qué forma tienen los datos? | `geom_point()`, `geom_line()`, `geom_col()`... |
| **Estadística** | ¿Hay que transformar antes de dibujar? | `stat_summary()`, `geom_smooth()`, `geom_histogram()` |
| **Escalas** | ¿Cómo se traduce el dato a posición/color? | `scale_x_log10()`, `scale_colour_viridis_c()` |
| **Coordenadas** | ¿Qué sistema de coordenadas? | `coord_flip()`, `coord_polar()`, `coord_sf()` |
| **Facetas** | ¿Un panel por grupo? | `facet_wrap()`, `facet_grid()` |
| *(+ Tema)* | ¿Cómo se ve lo que no son datos? | `theme_minimal()`, `theme()` |

## Datos *tidy*: la forma que ggplot2 espera

ggplot2 quiere **una columna por variable y una fila por observación**. Muchos datos llegan "anchos" (un año por columna); `pivot_longer()` los convierte a "largos".

In [ ]:
ancho <- tibble(ciudad = c("Bogotá", "Medellín", "Cali"),
                `2023` = c(120, 95, 60), `2024` = c(135, 100, 72), `2025` = c(150, 118, 80))
ancho

largo <- ancho |> pivot_longer(cols = -ciudad, names_to = "anio", values_to = "ventas")
largo

In [ ]:
options(repr.plot.height = 4)
ggplot(largo, aes(x = anio, y = ventas, fill = ciudad)) +
  geom_col(position = "dodge") +
  scale_fill_manual(values = paleta) +
  labs(title = "Con datos largos, ggplot2 agrupa y colorea solo", x = NULL, y = "Ventas")
options(repr.plot.height = 5)

## Estéticas: *mapear* una variable vs *fijar* un valor

Regla de oro: **dentro** de `aes()` va una **variable** (ggplot2 crea la leyenda); **fuera** de `aes()` va un **valor fijo**.

In [ ]:
options(repr.plot.height = 4)
p_mapea <- ggplot(mpg, aes(displ, hwy, colour = drv)) + geom_point() +
  labs(title = "colour DENTRO de aes(): mapea la variable drv")
p_fija  <- ggplot(mpg, aes(displ, hwy)) + geom_point(colour = verde) +
  labs(title = "colour FUERA de aes(): un valor fijo")
p_mapea | p_fija
options(repr.plot.height = 5)

In [ ]:
options(repr.plot.height = 4.5)
# Varias estéticas a la vez: color, tamaño y forma codifican tres variables más
ggplot(mpg, aes(displ, hwy, colour = drv, size = cyl, shape = fl)) +
  geom_point(alpha = 0.6) +
  scale_colour_manual(values = paleta) +
  labs(title = "Cuidado: más de 3 estéticas suele ser ilegible",
       colour = "Tracción", size = "Cilindros", shape = "Combustible")
options(repr.plot.height = 5)

# Capítulo 2 · Catálogo de geometrías según la pregunta

La pregunta correcta no es "¿qué gráfico hago?" sino **"¿qué tipo de variables tengo y qué quiero mostrar?"**.

## Una variable numérica: ¿cómo se distribuye?

In [ ]:
options(repr.plot.height = 7)
h1 <- ggplot(diamonds, aes(price)) + geom_histogram(bins = 50, fill = verde, colour = "white") +
  labs(title = "geom_histogram(bins = 50)")
h2 <- ggplot(diamonds, aes(price)) + geom_density(fill = verde, alpha = 0.4, colour = verde) +
  labs(title = "geom_density()")
h3 <- ggplot(diamonds, aes(price, colour = cut)) + geom_freqpoly(bins = 40, linewidth = 0.8) +
  scale_colour_manual(values = paleta) + labs(title = "geom_freqpoly() por grupo")
h4 <- ggplot(diamonds, aes(price)) + geom_histogram(bins = 50, fill = ambar, colour = "white") +
  scale_x_log10(labels = label_dollar()) + labs(title = "+ scale_x_log10(): distribuciones sesgadas")
(h1 | h2) / (h3 | h4)
options(repr.plot.height = 5)

## Una variable categórica: ¿cuántos hay en cada grupo?

In [ ]:
options(repr.plot.height = 7)
b1 <- ggplot(mpg, aes(class)) + geom_bar(fill = verde) +
  labs(title = "geom_bar(): cuenta las filas por ti")
b2 <- mpg |> count(class) |> mutate(class = fct_reorder(class, n)) |>
  ggplot(aes(n, class)) + geom_col(fill = ambar) +
  labs(title = "geom_col() + fct_reorder(): ordenado y horizontal", y = NULL)
b3 <- mpg |> count(class, drv) |>
  ggplot(aes(n, fct_reorder(class, n, sum), fill = drv)) + geom_col() +
  scale_fill_manual(values = paleta) + labs(title = "position = 'stack' (por defecto)", y = NULL)
b4 <- mpg |> count(class, drv) |>
  ggplot(aes(n, fct_reorder(class, n, sum), fill = drv)) + geom_col(position = "fill") +
  scale_fill_manual(values = paleta) + scale_x_continuous(labels = label_percent()) +
  labs(title = "position = 'fill': proporciones", y = NULL, x = NULL)
(b1 | b2) / (b3 | b4)
options(repr.plot.height = 5)

> **Tip:** ordena siempre las barras por valor (`fct_reorder`) salvo que el orden tenga significado (meses, niveles educativos...).

## Dos variables numéricas: ¿se relacionan?

In [ ]:
options(repr.plot.height = 7)
s1 <- ggplot(mpg, aes(cty, hwy)) + geom_point() +
  labs(title = "geom_point(): ¿ves la sobreposición?")
s2 <- ggplot(mpg, aes(cty, hwy)) + geom_jitter(width = 0.3, height = 0.3, alpha = 0.5, colour = verde) +
  labs(title = "geom_jitter() + alpha: la solución")
s3 <- ggplot(diamonds, aes(carat, price)) + geom_hex(bins = 40) + scale_fill_viridis_c() +
  labs(title = "geom_hex(): 54 mil puntos sin saturar")
s4 <- ggplot(mpg, aes(displ, hwy)) + geom_point(alpha = 0.4) +
  geom_smooth(method = "lm", formula = y ~ x, colour = ambar) + geom_rug(alpha = 0.2) +
  labs(title = "geom_smooth(method = 'lm') + geom_rug()")
(s1 | s2) / (s3 | s4)
options(repr.plot.height = 5)

## Numérica × categórica: ¿cómo se compara la distribución entre grupos?

In [ ]:
options(repr.plot.height = 7)
v1 <- ggplot(mpg, aes(drv, hwy)) + geom_boxplot(fill = "#CFE8DF") +
  labs(title = "geom_boxplot()")
v2 <- ggplot(mpg, aes(drv, hwy, fill = drv)) + geom_violin(alpha = 0.6, show.legend = FALSE) +
  geom_jitter(width = 0.1, alpha = 0.3) + scale_fill_manual(values = paleta) +
  labs(title = "geom_violin() + geom_jitter()")
v3 <- ggplot(mpg, aes(hwy, class, fill = class)) + geom_density_ridges(alpha = 0.7, show.legend = FALSE) +
  scale_fill_manual(values = paleta) + labs(title = "ggridges::geom_density_ridges()", y = NULL)
v4 <- ggplot(mpg, aes(class, hwy)) +
  stat_summary(fun = mean, geom = "point", size = 3, colour = verde) +
  stat_summary(fun.data = mean_se, geom = "errorbar", width = 0.2, colour = verde) +
  labs(title = "stat_summary(): media ± error estándar")
(v1 | v2) / (v3 | v4)
options(repr.plot.height = 5)

## Series de tiempo: ¿cómo evoluciona?

In [ ]:
options(repr.plot.height = 7)
ts1 <- ggplot(economics, aes(date, unemploy)) + geom_line(colour = verde) +
  labs(title = "geom_line()", y = "Desempleados (miles)")
ts2 <- ggplot(economics, aes(date, unemploy)) + geom_area(fill = verde, alpha = 0.4) +
  labs(title = "geom_area()", y = NULL)
econ_largo <- economics |> select(date, psavert, uempmed) |>
  pivot_longer(-date, names_to = "indicador", values_to = "valor")
ts3 <- ggplot(econ_largo, aes(date, valor, colour = indicador)) + geom_line() +
  scale_colour_manual(values = paleta) +
  scale_x_date(date_breaks = "10 years", date_labels = "%Y") +
  labs(title = "Varias series: formato largo + colour", x = NULL, y = NULL)
ts4 <- economics |> filter(date >= as.Date("2005-01-01")) |>
  ggplot(aes(date, unemploy / 1000)) +
  geom_ribbon(aes(ymin = 0, ymax = unemploy / 1000), fill = ambar, alpha = 0.3) +
  geom_line(colour = "#B7791F") +
  geom_vline(xintercept = as.Date("2008-09-15"), linetype = "dashed") +
  annotate("text", x = as.Date("2008-12-01"), y = 14.5, label = "Crisis 2008", hjust = 0) +
  labs(title = "geom_ribbon() + geom_vline() + annotate()", x = NULL, y = "Millones")
(ts1 | ts2) / (ts3 | ts4)
options(repr.plot.height = 5)

## Dos categóricas: ¿cómo se cruzan?

In [ ]:
options(repr.plot.height = 7)
c1 <- ggplot(mpg, aes(class, drv)) + geom_count(colour = verde) +
  labs(title = "geom_count(): tamaño = frecuencia")
c2 <- mpg |> count(class, drv) |>
  ggplot(aes(class, drv, fill = n)) + geom_tile(colour = "white") +
  geom_text(aes(label = n), colour = "white", fontface = "bold") +
  scale_fill_viridis_c(guide = "none") + labs(title = "geom_tile() + geom_text(): mapa de calor")
c3 <- ggplot(mpg, aes(class, fill = drv)) + geom_bar(position = "dodge") +
  scale_fill_manual(values = paleta) + labs(title = "geom_bar(position = 'dodge')")
c4 <- ggplot(diamonds, aes(cut, fill = clarity)) + geom_bar(position = "fill") +
  scale_fill_viridis_d(option = "E") + scale_y_continuous(labels = label_percent()) +
  labs(title = "position = 'fill' con una escala viridis discreta", y = NULL)
(c1 | c2) / (c3 | c4)
options(repr.plot.height = 5)

## Muchas variables: correlaciones y pares

In [ ]:
options(repr.plot.height = 5.5)
matriz <- cor(select(mtcars, mpg, cyl, disp, hp, drat, wt, qsec))
ggcorrplot(matriz, type = "lower", lab = TRUE, lab_size = 3,
           colors = c("#B8412D", "white", verde),
           title = "ggcorrplot(): correlaciones de mtcars")
options(repr.plot.height = 5)

In [ ]:
options(repr.plot.height = 6)
pairs(select(mtcars, mpg, disp, hp, wt), col = adjustcolor(verde, 0.6), pch = 19,
      main = "pairs(): matriz de dispersión (base R). Alternativa: GGally::ggpairs()")
options(repr.plot.height = 5)

## Texto y etiquetas sobre el gráfico

In [ ]:
options(repr.plot.height = 5)
destacados <- mpg |> filter(hwy >= 41)
ggplot(mpg, aes(displ, hwy)) +
  geom_point(alpha = 0.4) +
  geom_label_repel(data = destacados, aes(label = paste(manufacturer, model)),
                   fill = "#FFF3D6", size = 3.2, min.segment.length = 0) +
  labs(title = "ggrepel::geom_label_repel(): etiqueta solo lo importante y evita superposiciones")
options(repr.plot.height = 5)

## Mapas: datos geoespaciales con `sf`

`sf` guarda geometrías (polígonos, puntos) en una columna del data frame; `geom_sf()` las dibuja. Aquí usamos un mapa de ejemplo incluido en el paquete (condados de Carolina del Norte).

In [ ]:
options(repr.plot.height = 4.5)
nc <- sf::st_read(system.file("shape/nc.shp", package = "sf"), quiet = TRUE)
ggplot(nc) +
  geom_sf(aes(fill = BIR74), colour = "white", linewidth = 0.2) +
  scale_fill_viridis_c(option = "G", labels = label_comma()) +
  labs(title = "geom_sf(): mapa coroplético",
       subtitle = "Nacimientos en 1974 por condado (datos de ejemplo del paquete sf)", fill = "Nacimientos") +
  theme_void()
options(repr.plot.height = 5)

> Para mapas de Colombia o del mundo: `rnaturalearth` (fronteras), `geodata`/`GADM` (departamentos y municipios), `tmap` (mapas temáticos) y `leaflet` (interactivos, ver Capítulo 3).

# Capítulo 2 · Escalas, color, coordenadas y facetas

## Escalas: cómo se traduce el dato a posición y color

In [ ]:
options(repr.plot.height = 7)
muestra <- slice_sample(diamonds, n = 6000)
e1 <- ggplot(muestra, aes(carat, price)) + geom_point(alpha = 0.2, colour = verde) +
  scale_y_continuous(labels = label_dollar()) + scale_x_continuous(breaks = 0:5) +
  labs(title = "labels = label_dollar(), breaks = 0:5")
e2 <- ggplot(muestra, aes(carat, price)) + geom_point(alpha = 0.2, colour = verde) +
  scale_x_log10() + scale_y_log10(labels = label_dollar()) +
  labs(title = "scale_x_log10() + scale_y_log10(): relación lineal")
e3 <- ggplot(mpg, aes(displ, hwy)) + geom_point() +
  geom_smooth(method = "lm", formula = y ~ x, colour = ambar) +
  scale_y_continuous(limits = c(20, 35)) +
  labs(title = "limits = c(20, 35) ELIMINA datos: cambia la recta")
e4 <- ggplot(mpg, aes(displ, hwy)) + geom_point() +
  geom_smooth(method = "lm", formula = y ~ x, colour = ambar) +
  coord_cartesian(ylim = c(20, 35)) +
  labs(title = "coord_cartesian(ylim = ...) solo hace ZOOM")
(e1 | e2) / (e3 | e4)
options(repr.plot.height = 5)

## Color con propósito

Tres tipos de paleta según el tipo de variable, y una técnica para **resaltar**:

In [ ]:
options(repr.plot.height = 7.5)
okabe_ito <- unname(palette.colors(palette = "Okabe-Ito"))   # paleta accesible incluida en R
k1 <- ggplot(mpg, aes(displ, hwy, colour = class)) + geom_point(size = 2) +
  scale_colour_manual(values = okabe_ito) + labs(title = "Categórica: Okabe-Ito (apta para daltonismo)")
k2 <- ggplot(mpg, aes(displ, hwy, colour = cty)) + geom_point(size = 2) +
  scale_colour_viridis_c() + labs(title = "Secuencial: viridis (perceptualmente uniforme)")
k3 <- mtcars |> mutate(auto = rownames(mtcars), dif = mpg - mean(mpg), auto = fct_reorder(auto, dif)) |>
  ggplot(aes(dif, auto, fill = dif)) + geom_col(show.legend = FALSE) +
  scale_fill_gradient2(low = "#B8412D", mid = "grey90", high = verde) +
  labs(title = "Divergente: scale_fill_gradient2()", x = "mpg vs promedio", y = NULL) +
  theme(axis.text.y = element_text(size = 7))
k4 <- mpg |> mutate(destacar = manufacturer == "toyota") |>
  ggplot(aes(displ, hwy, colour = destacar)) + geom_point(size = 2) +
  scale_colour_manual(values = c("grey80", verde), labels = c("Otros", "Toyota"), name = NULL) +
  labs(title = "Resaltar: todo gris + un color")
(k1 | k2) / (k3 | k4)
options(repr.plot.height = 5)

In [ ]:
options(repr.plot.height = 8)
# RColorBrewer trae paletas clásicas: secuenciales (arriba), cualitativas (medio) y divergentes (abajo)
RColorBrewer::display.brewer.all()
# Se usan con scale_colour_brewer(palette = "Dark2") o scale_fill_brewer(palette = "YlGnBu")
options(repr.plot.height = 5)

## Coordenadas

In [ ]:
options(repr.plot.height = 7)
co1 <- ggplot(mpg, aes(class)) + geom_bar(fill = verde) + coord_flip() + labs(title = "coord_flip()")
co2 <- mpg |> count(drv) |>
  ggplot(aes(x = "", y = n, fill = drv)) + geom_col(width = 1) + coord_polar(theta = "y") +
  scale_fill_manual(values = paleta) + theme_void() +
  labs(title = "coord_polar(): pastel (úsalo solo con 2-3 categorías)")
co3 <- ggplot(mpg, aes(cty, hwy)) + geom_point(alpha = 0.4) + coord_fixed() +
  geom_abline(slope = 1, intercept = 0, linetype = "dashed") + labs(title = "coord_fixed(): misma escala en x e y")
co4 <- ggplot(mpg, aes(displ, hwy)) + geom_point() + coord_cartesian(xlim = c(2, 4), ylim = c(20, 35)) +
  labs(title = "coord_cartesian(): zoom sin perder datos")
(co1 | co2) / (co3 | co4)
options(repr.plot.height = 5)

## Facetas: *small multiples*

Comparar grupos en paneles separados con **los mismos ejes** es una de las técnicas más potentes de la visualización.

In [ ]:
options(repr.plot.height = 12)
fa1 <- ggplot(mpg, aes(displ, hwy)) + geom_point(alpha = 0.5, colour = verde) +
  facet_wrap(~ class, ncol = 4) + labs(title = "facet_wrap(~ class): un panel por categoría")
fa2 <- ggplot(mpg, aes(displ, hwy)) + geom_point(alpha = 0.5, colour = verde) +
  facet_grid(drv ~ cyl, labeller = label_both) + labs(title = "facet_grid(drv ~ cyl): filas × columnas")
fa3 <- ggplot(economics_long, aes(date, value)) + geom_line(colour = verde) +
  facet_wrap(~ variable, scales = "free_y", ncol = 5) +
  labs(title = "scales = 'free_y': cada panel con su propio eje (útil con magnitudes distintas)", x = NULL, y = NULL)
fa1 / fa2 / fa3
options(repr.plot.height = 5)

## Estadísticas dentro del gráfico

In [ ]:
options(repr.plot.height = 7)
st1 <- ggplot(mpg, aes(hwy, colour = drv)) + stat_ecdf(linewidth = 1) +
  scale_colour_manual(values = paleta) + labs(title = "stat_ecdf(): distribución acumulada", y = "Proporción")
st2 <- ggplot(mpg, aes(displ, hwy)) + geom_point(alpha = 0.3) +
  geom_smooth(method = "lm", formula = y ~ x, colour = ambar, se = FALSE) +
  geom_smooth(method = "loess", formula = y ~ x, colour = verde, se = FALSE) +
  labs(title = "geom_smooth(): lm (ámbar) vs loess (verde)")
st3 <- ggplot(diamonds, aes(carat, price)) + geom_bin2d(bins = 40) + scale_fill_viridis_c() +
  labs(title = "geom_bin2d(): densidad en rejilla")
st4 <- ggplot(mpg, aes(class, hwy)) + stat_summary(fun = median, geom = "bar", fill = verde) +
  labs(title = "stat_summary(fun = median, geom = 'bar')")
(st1 | st2) / (st3 | st4)
options(repr.plot.height = 5)

# Capítulo 2 · Del gráfico exploratorio al comunicativo

## Títulos, etiquetas y anotaciones

Un gráfico comunicativo tiene **un título que afirma el hallazgo**, ejes con unidades, fuente y anotaciones que guían la mirada (recuerda la *Guía de títulos, gráficos y color* del curso).

In [ ]:
options(repr.plot.height = 5.5)
ggplot(mpg, aes(displ, hwy)) +
  annotate("rect", xmin = 1.5, xmax = 2.2, ymin = 40, ymax = 46, fill = ambar, alpha = 0.2) +
  geom_point(colour = verde, alpha = 0.6, size = 2) +
  geom_hline(yintercept = mean(mpg$hwy), linetype = "dashed", colour = "grey40") +
  annotate("text", x = 6.6, y = mean(mpg$hwy) + 1.2, label = "Promedio", colour = "grey40", size = 3.5) +
  annotate("curve", x = 3.3, y = 44, xend = 2.25, yend = 44, curvature = 0.3,
           arrow = arrow(length = unit(0.2, "cm")), colour = "#B7791F") +
  annotate("text", x = 3.4, y = 44, label = "Híbridos: la excepción", hjust = 0, fontface = "italic", colour = "#B7791F") +
  labs(title = "A mayor cilindrada, menor rendimiento en carretera",
       subtitle = "234 modelos (1999-2008); los híbridos rompen la tendencia",
       x = "Cilindrada del motor (litros)", y = "Rendimiento en carretera (millas por galón)",
       caption = "Fuente: EPA vía ggplot2::mpg | Elaboración propia", tag = "A") +
  theme(plot.title.position = "plot")
options(repr.plot.height = 5)

## Temas: lo que no son datos

In [ ]:
options(repr.plot.height = 7)
p <- ggplot(mpg, aes(displ, hwy, colour = drv)) + geom_point() + scale_colour_manual(values = paleta)
(p + theme_gray()    + labs(title = "theme_gray() (por defecto)")) +
(p + theme_bw()      + labs(title = "theme_bw()")) +
(p + theme_minimal() + labs(title = "theme_minimal()")) +
(p + theme_classic() + labs(title = "theme_classic()")) +
(p + theme_light()   + labs(title = "theme_light()")) +
(p + theme_void()    + labs(title = "theme_void()")) +
  plot_layout(ncol = 3, guides = "collect")
options(repr.plot.height = 5)

`theme()` controla cada elemento con `element_text()`, `element_line()`, `element_rect()` y `element_blank()`. Lo más útil es **definir un tema propio** y reutilizarlo:

In [ ]:
options(repr.plot.height = 5)
tema_ean <- function(base_size = 13) {
  theme_minimal(base_size = base_size) +
    theme(plot.title = element_text(face = "bold", colour = verde_oscuro, size = rel(1.25)),
          plot.subtitle = element_text(colour = gris),
          plot.caption = element_text(colour = gris, size = rel(0.75), hjust = 0),
          plot.title.position = "plot",
          panel.grid.minor = element_blank(),
          panel.grid.major.x = element_blank(),
          legend.position = "top",
          axis.title = element_text(colour = gris),
          strip.text = element_text(face = "bold", colour = verde_oscuro))
}

p + tema_ean() +
  labs(title = "Un tema propio se aplica con + como cualquier capa",
       subtitle = "tema_ean() encapsula las decisiones de estilo del curso", caption = "Fuente: ggplot2::mpg")

theme_set(tema_ean())   # desde aquí, todos los gráficos del cuaderno lo usan
options(repr.plot.height = 5)

## Componer varios gráficos: patchwork

In [ ]:
options(repr.plot.height = 6.5)
pa <- ggplot(mpg, aes(hwy)) + geom_density(fill = verde, alpha = 0.5, colour = verde) + labs(title = "Distribución de hwy")
pb <- ggplot(mpg, aes(cty, hwy)) + geom_point(colour = ambar) + labs(title = "cty vs hwy")
pc <- mpg |> count(manufacturer) |> mutate(manufacturer = fct_reorder(manufacturer, n)) |>
  ggplot(aes(n, manufacturer)) + geom_col(fill = verde) + labs(title = "Modelos por fabricante", y = NULL)

(pa | pb) / pc +
  plot_annotation(tag_levels = "A", title = "(pa | pb) / pc: | pone al lado, / pone debajo")
options(repr.plot.height = 5)

In [ ]:
options(repr.plot.height = 4.5)
# Un gráfico dentro de otro
pb + inset_element(pa + theme_void(), left = 0.55, bottom = 0.55, right = 1, top = 1)
options(repr.plot.height = 5)

## Exportar: `ggsave()`

In [ ]:
g <- p + labs(title = "Gráfico exportado desde R", caption = "Fuente: ggplot2::mpg")
ggsave("grafico_clase18.png", g, width = 8, height = 5, dpi = 300, bg = "white")   # para diapositivas e informes
ggsave("grafico_clase18.pdf", g, width = 8, height = 5)                             # vectorial para imprimir
ggsave("grafico_clase18.svg", g, width = 8, height = 5)                             # vectorial para web (svglite)
list.files(pattern = "grafico_clase18")

| Destino | Formato | Tamaño sugerido |
|---|---|---|
| Diapositiva 16:9 | PNG 300 dpi | 10 × 5.6 pulgadas |
| Informe / artículo | PDF o SVG | 7 × 4.5 pulgadas |
| Web / dashboard | PNG 150 dpi o widget HTML | 9 × 5 pulgadas |

# Capítulo 3 · Más allá de lo estático: interactividad

## De ggplot2 a interactivo en una línea: `ggplotly()`

In [ ]:
library(plotly)
p_int <- ggplot(mpg, aes(displ, hwy, colour = drv,
                         text = paste0(manufacturer, " ", model, " (", year, ")"))) +
  geom_point(size = 2.5, alpha = 0.8) +
  scale_colour_manual(values = paleta[1:3], name = "Tracción") +
  labs(title = "Pasa el mouse, haz zoom, arrastra, descarga", x = "Cilindrada (L)", y = "mpg carretera")
ggplotly(p_int, tooltip = c("text", "x", "y"))

## `plot_ly()` nativo y tablas interactivas con `DT`

In [ ]:
plot_ly(economics, x = ~date, y = ~unemploy, type = "scatter", mode = "lines",
        line = list(color = verde, width = 2)) |>
  plotly::layout(title = "plot_ly(): sintaxis propia de plotly, con rango deslizable",
                 xaxis = list(rangeslider = list(visible = TRUE)), yaxis = list(title = "Desempleados (miles)"))

In [ ]:
DT::datatable(head(mpg, 40), filter = "top", options = list(pageLength = 6),
              caption = "DT::datatable(): tabla con búsqueda, filtros y orden")

## Mapas interactivos con `leaflet` (opcional)

In [ ]:
if (requireNamespace("leaflet", quietly = TRUE)) {
  library(leaflet)
  sedes <- tibble(nombre = "Universidad EAN (Bogotá, ubicación aproximada)", lat = 4.664, lng = -74.055)
  leaflet(sedes) |> addTiles() |> addMarkers(~lng, ~lat, popup = ~nombre) |> setView(-74.055, 4.664, zoom = 14)
} else {
  cat("Instala el paquete con install.packages('leaflet') para ver el mapa interactivo.\n")
}

## Otros paquetes interactivos (opcionales)

| Paquete | Qué hace | Una línea |
|---|---|---|
| `highcharter` | Gráficos Highcharts con animaciones | `hchart(mpg, "scatter", hcaes(displ, hwy, group = drv))` |
| `dygraphs` | Series de tiempo con zoom | `dygraph(serie_ts)` |
| `echarts4r` | Gráficos ECharts (incluye 3D y mapas) | `mpg |> e_charts(displ) |> e_scatter(hwy)` |
| `ggiraph` | ggplot2 con tooltips SVG | `girafe(ggobj = p)` |
| `visNetwork` / `networkD3` | Redes interactivas | `visNetwork(nodes, edges)` |
| `gganimate` | Animaciones a partir de ggplot2 | ver abajo |

```r
# Requiere: install.packages(c("gganimate", "gifski"))
library(gganimate)
anim <- ggplot(economics, aes(date, unemploy / 1000)) +
  geom_line(colour = verde, linewidth = 1) +
  labs(title = "Desempleo en EE. UU.: la línea se dibuja con el tiempo", y = "Millones") +
  transition_reveal(date)
animate(anim, nframes = 120, fps = 20, width = 800, height = 450, renderer = gifski_renderer())
anim_save("desempleo.gif")
```

```r
# Cualquier widget (plotly, leaflet, DT...) se guarda como página HTML independiente:
htmlwidgets::saveWidget(ggplotly(p_int), "grafico_interactivo.html", selfcontained = TRUE)
```

# Capítulo 3 · Dashboards y reportes reproducibles

## R Markdown y Quarto: este cuaderno es un reporte

Este mismo archivo se convierte en HTML, PDF o Word con **Knit** (R Markdown) o `quarto render` (Quarto). Texto + código + resultados en un solo documento **reproducible**: si cambian los datos, cambian los gráficos.

## flexdashboard: un dashboard en un archivo `.Rmd`

Crea un archivo nuevo con *File → New File → R Markdown → From Template → Flex Dashboard* y organiza los gráficos con encabezados (cada `###` es una tarjeta, cada `Column` una columna):

```
---
title: "Mercado de vivienda en Texas"
output: flexdashboard::flex_dashboard
---

Column {data-width=600}
-----------------------------------------------------------------------

### Ventas por año

In [ ]:
ggplotly(p_int)

Column {data-width=400}
-----------------------------------------------------------------------

### Tabla de datos

In [ ]:
DT::datatable(txhousing)

```

## Shiny: aplicaciones web reactivas

Una app Shiny tiene **dos partes**: `ui` (qué ve el usuario) y `server` (qué calcula R). La **reactividad** conecta ambas: cuando cambia un *input*, se recalculan solo los *outputs* que dependen de él.

```r
# Requiere: install.packages("shiny"). Guarda esto como app.R y pulsa "Run App".
library(shiny); library(tidyverse)

ui <- fluidPage(
  titlePanel("Explorador de rendimiento de autos (mpg)"),
  sidebarLayout(
    sidebarPanel(
      selectInput("clase", "Tipo de vehículo", choices = sort(unique(mpg$class)), selected = "suv"),
      sliderInput("alpha", "Transparencia de los puntos", min = 0.1, max = 1, value = 0.6)
    ),
    mainPanel(plotOutput("grafico"), tableOutput("resumen"))
  )
)

server <- function(input, output) {
  datos <- reactive(filter(mpg, class == input$clase))          # se recalcula al cambiar "clase"
  output$grafico <- renderPlot({
    ggplot(datos(), aes(displ, hwy)) +
      geom_point(alpha = input$alpha, size = 3, colour = "#0B6E4F") +
      labs(title = paste("Clase:", input$clase), x = "Cilindrada (L)", y = "mpg carretera")
  })
  output$resumen <- renderTable(datos() |> summarise(modelos = n(), hwy_promedio = round(mean(hwy), 1)))
}

shinyApp(ui, server)
```

**Despliegue:** `shinyapps.io` (gratis para empezar), Posit Connect, un servidor propio con Shiny Server, o **Shinylive** (la app corre en el navegador sin servidor).

| Necesito... | Herramienta |
|---|---|
| Un informe que se regenera con datos nuevos | R Markdown / Quarto |
| Un panel de gráficos interactivos sin lógica compleja | flexdashboard / Quarto dashboards |
| Filtros, escenarios "¿qué pasa si...?", cálculos en vivo | Shiny |

# Capítulo 4 · Flujo completo de análisis: caso práctico

Aplicaremos las **cinco fases** del análisis exploratorio con el mercado de vivienda de Texas (`txhousing`): 8 602 registros mensuales de 46 ciudades entre 2000 y 2015.

## Definir la pregunta y obtener los datos

> **Pregunta:** ¿cómo evolucionaron las ventas de vivienda en las cuatro grandes ciudades de Texas y qué tan estacional es el mercado?

```r
# Si los datos vinieran de fuera, así se importan:
datos <- readr::read_csv("ventas_vivienda.csv")                  # CSV
datos <- readxl::read_excel("ventas.xlsx", sheet = 1)            # Excel
datos <- jsonlite::fromJSON("https://api.ejemplo.org/ventas")    # API JSON
datos <- DBI::dbGetQuery(conexion, "SELECT * FROM ventas")       # base de datos
```

In [ ]:
datos <- txhousing
glimpse(datos)

## Limpiar y transformar con dplyr y tidyr

In [ ]:
# ¿Cuántos valores faltantes hay por columna?
colSums(is.na(datos))

grandes <- c("Houston", "Dallas", "Austin", "San Antonio")
meses <- c("Ene", "Feb", "Mar", "Abr", "May", "Jun", "Jul", "Ago", "Sep", "Oct", "Nov", "Dic")

# ¿Todos los años están completos? Si un año tiene menos de 12 meses, un total anual engaña
datos |> filter(city == "Houston") |> count(year) |> filter(n < 12)

2015 solo llega hasta julio: si lo sumáramos, parecería un desplome de ventas que no existió. **Antes de graficar, revisa la cobertura de los datos.**

In [ ]:
resumen_anual <- datos |>
  filter(city %in% grandes, year <= 2014) |>      # excluimos el año incompleto
  group_by(city, year) |>
  summarise(ventas = sum(sales, na.rm = TRUE),
            precio_mediano = median(median, na.rm = TRUE), .groups = "drop")
resumen_anual

## Visualizar patrones

In [ ]:
options(repr.plot.height = 5)
ggplot(resumen_anual, aes(year, ventas / 1000, colour = city)) +
  geom_line(linewidth = 1.1) + geom_point(size = 1.8) +
  scale_colour_manual(values = paleta, name = NULL) +
  scale_x_continuous(breaks = seq(2000, 2014, 2)) +
  labs(title = "Houston lidera las ventas; las cuatro ciudades cayeron con la crisis de 2008",
       subtitle = "Ventas anuales de vivienda, 2000-2014", x = NULL, y = "Ventas (miles)",
       caption = "Fuente: ggplot2::txhousing (TAMU Real Estate Center)")
options(repr.plot.height = 5)

In [ ]:
options(repr.plot.height = 4.5)
estacional <- datos |> filter(city == "Houston") |>
  group_by(month) |> summarise(ventas = mean(sales, na.rm = TRUE))
razon <- estacional$ventas[6] / estacional$ventas[1]     # junio vs enero
ggplot(estacional, aes(month, ventas)) +
  geom_col(fill = verde) +
  scale_x_continuous(breaks = 1:12, labels = meses) +
  labs(title = paste0("El mercado es estacional: junio vende ", round((razon - 1) * 100), " % más que enero"),
       subtitle = "Promedio mensual de ventas en Houston, 2000-2015", x = NULL, y = "Ventas promedio")
options(repr.plot.height = 5)

In [ ]:
options(repr.plot.height = 5.5)
datos |> filter(city == "Houston") |>
  ggplot(aes(month, factor(year), fill = sales)) +
  geom_tile(colour = "white") +
  scale_fill_viridis_c(labels = label_comma(), name = "Ventas") +
  scale_x_continuous(breaks = 1:12, labels = meses, expand = c(0, 0)) +
  labs(title = "Dos patrones a la vez: estacionalidad (columnas) y tendencia (filas)",
       subtitle = "Ventas mensuales en Houston", x = NULL, y = NULL) +
  theme(panel.grid = element_blank(), legend.position = "right")
options(repr.plot.height = 5)

In [ ]:
options(repr.plot.height = 5)
ggplot(filter(datos, city %in% grandes), aes(listings, sales)) +
  geom_point(alpha = 0.3, colour = verde) +
  geom_smooth(method = "lm", formula = y ~ x, colour = ambar) +
  facet_wrap(~ city, scales = "free") +
  scale_x_continuous(labels = label_comma()) +
  labs(title = "Más inmuebles listados, más ventas: la relación se repite en las cuatro ciudades",
       x = "Inmuebles en venta (listings)", y = "Ventas mensuales")
options(repr.plot.height = 5)

## Modelar relaciones

In [ ]:
houston <- filter(datos, city == "Houston")
modelo <- lm(sales ~ listings + factor(month), data = houston)
cat("R² del modelo:", round(summary(modelo)$r.squared, 3), "\n")
round(coef(modelo)[1:2], 3)

In [ ]:
options(repr.plot.height = 4.5)
houston |> mutate(prediccion = predict(modelo, houston)) |>
  ggplot(aes(date)) +
  geom_line(aes(y = sales, colour = "Observado"), linewidth = 0.8) +
  geom_line(aes(y = prediccion, colour = "Modelo"), linewidth = 0.8, linetype = "dashed") +
  scale_colour_manual(values = c(Observado = verde, Modelo = ambar), name = NULL) +
  labs(title = "Un modelo lineal con estacionalidad reproduce bien las ventas de Houston",
       subtitle = "sales ~ listings + factor(month)", x = NULL, y = "Ventas mensuales")
options(repr.plot.height = 5)

## Comunicar resultados

In [ ]:
options(repr.plot.height = 5.5)
# ¿En qué año recuperó cada ciudad su máximo anterior a la crisis?
recuperacion <- resumen_anual |> group_by(city) |>
  mutate(pico_previo = max(ventas[year <= 2007])) |>
  filter(year > 2009, ventas >= pico_previo) |>
  summarise(anio = min(year))
recuperacion

# El título se construye a partir del resultado: así nunca afirma algo que los datos no digan
titulo <- if (nrow(recuperacion) < length(grandes)) {
  paste0("Tras la crisis de 2008, solo ",
         paste(paste0(recuperacion$city, " (", recuperacion$anio, ")"), collapse = " y "),
         " superaron su máximo previo")
} else {
  paste0("Tras la crisis de 2008, las cuatro ciudades superaron su máximo previo entre ",
         min(recuperacion$anio), " y ", max(recuperacion$anio))
}

final <- ggplot(resumen_anual, aes(year, ventas / 1000, colour = city)) +
  annotate("rect", xmin = 2007.5, xmax = 2010.5, ymin = -Inf, ymax = Inf, fill = "grey85", alpha = 0.5) +
  annotate("text", x = 2009, y = max(resumen_anual$ventas) / 1000, label = "Crisis financiera", size = 3.3, colour = gris) +
  geom_line(linewidth = 1.2) +
  geom_text_repel(data = filter(resumen_anual, year == 2014), aes(label = city),
                  nudge_x = 0.5, direction = "y", hjust = 0, size = 3.5, segment.colour = NA) +
  scale_colour_manual(values = paleta, guide = "none") +
  scale_x_continuous(breaks = seq(2000, 2014, 2), limits = c(2000, 2016.5)) +
  labs(title = titulo,
       subtitle = "Ventas anuales de vivienda en las cuatro mayores ciudades de Texas, 2000-2014",
       x = NULL, y = "Ventas (miles)",
       caption = "Fuente: Texas A&M Real Estate Center vía ggplot2::txhousing | Elaboración propia")
final
ggsave("ventas_texas_final.png", final, width = 10, height = 5.6, dpi = 300, bg = "white")
options(repr.plot.height = 5)

# Buenas prácticas: lista de chequeo antes de publicar un gráfico

- [ ] El **título afirma el hallazgo** (no "Ventas por año" sino "Las ventas se duplicaron desde 2011").
- [ ] Los ejes tienen **nombre y unidades**; los números usan separadores (`label_comma()`, `label_percent()`).
- [ ] El **color tiene un propósito**: categórico, secuencial o divergente; paleta accesible (viridis, Okabe-Ito).
- [ ] Un solo mensaje por gráfico; si hay muchos grupos, **facetas** o **resaltar** uno.
- [ ] Hay **fuente** de los datos (`caption`) y fecha.
- [ ] Sin *chartjunk*: fondos, rejillas menores, 3D y efectos innecesarios (`theme_minimal()` ayuda).
- [ ] El eje Y de las barras **empieza en cero**; las escalas logarítmicas se anuncian.
- [ ] Exportado en el tamaño y la resolución del medio (`ggsave()`).

# 🧪 Tu turno: ejercicios

Completa los espacios `___`. Cada ejercicio usa lo visto en una sección; al final hay soluciones sugeridas.

```r
# 1. Distribución del precio de los diamantes por tipo de corte (cut).
#    Usa geom_boxplot(), escala logarítmica en y con etiquetas de dólares y un título que afirme algo.
ggplot(diamonds, aes(x = ___, y = ___)) +
  geom_boxplot(fill = "#CFE8DF") +
  scale_y_log10(labels = ___) +
  labs(title = "___")
```

```r
# 2. Barras ordenadas: número de modelos por fabricante, coloreadas por tracción (drv), en horizontal.
mpg |> count(___, ___) |>
  ggplot(aes(x = n, y = fct_reorder(___, n, sum), fill = ___)) +
  geom_col(position = "___") +
  labs(y = NULL)
```

```r
# 3. iris: relación entre largo y ancho del pétalo por especie, con recta de regresión por grupo y facetas.
ggplot(iris, aes(Petal.Length, Petal.Width, colour = ___)) +
  geom_point() +
  geom_smooth(method = "___", formula = y ~ x, se = FALSE) +
  facet_wrap(~ ___)
```

```r
# 4. airquality: temperatura diaria (Temp) a lo largo del año, un panel por mes (Month), con la media como línea horizontal.
ggplot(airquality, aes(Day, Temp)) +
  geom_line(colour = verde) +
  geom_hline(yintercept = ___, linetype = "dashed") +
  facet_wrap(~ ___, ncol = 5) +
  labs(title = "___")
```

```r
# 5. Convierte el gráfico del ejercicio 3 en interactivo y muestra la especie en el tooltip.
g3 <- ggplot(iris, aes(Petal.Length, Petal.Width, colour = Species, text = ___)) + geom_point()
ggplotly(g3, tooltip = ___)
```

```r
# 6. Reto: reproduce el gráfico final del caso práctico para OTRAS cuatro ciudades de Texas
#    (por ejemplo "El Paso", "Fort Worth", "Corpus Christi", "Lubbock") y cambia el título por tu hallazgo.
mis_ciudades <- c("___", "___", "___", "___")
```

```r
# 7. Reto de diseño: toma cualquier gráfico del cuaderno y aplícale tema_ean(); luego crea tu propio tema
#    (cambia colores, fuente base y posición de la leyenda) y compáralos lado a lado con patchwork.
```

## Soluciones sugeridas

In [ ]:
options(repr.plot.height = 4.5)
ggplot(diamonds, aes(x = cut, y = price)) +
  geom_boxplot(fill = "#CFE8DF") +
  scale_y_log10(labels = label_dollar()) +
  labs(title = "Los diamantes de corte Ideal tienen el precio mediano más bajo: el corte no lo es todo",
       x = "Corte", y = "Precio (escala log)")
options(repr.plot.height = 5)

In [ ]:
options(repr.plot.height = 4.5)
mpg |> count(manufacturer, drv) |>
  ggplot(aes(x = n, y = fct_reorder(manufacturer, n, sum), fill = drv)) +
  geom_col(position = "stack") + scale_fill_manual(values = paleta) +
  labs(title = "Dodge y Toyota tienen más modelos; en Dodge predominan los 4x4", y = NULL, fill = "Tracción")
options(repr.plot.height = 5)

In [ ]:
options(repr.plot.height = 4)
ggplot(iris, aes(Petal.Length, Petal.Width, colour = Species)) +
  geom_point() + geom_smooth(method = "lm", formula = y ~ x, se = FALSE) +
  scale_colour_manual(values = paleta) + facet_wrap(~ Species)
options(repr.plot.height = 5)

In [ ]:
options(repr.plot.height = 4)
ggplot(airquality, aes(Day, Temp)) +
  geom_line(colour = verde) +
  geom_hline(yintercept = mean(airquality$Temp), linetype = "dashed") +
  facet_wrap(~ Month, ncol = 5) +
  labs(title = "Julio y agosto se mantienen por encima de la temperatura media del período (78 °F)")
options(repr.plot.height = 5)

In [ ]:
g3 <- ggplot(iris, aes(Petal.Length, Petal.Width, colour = Species, text = Species)) + geom_point() +
  scale_colour_manual(values = paleta)
ggplotly(g3, tooltip = c("text", "x", "y"))

# Chuleta: mapa del ecosistema de visualización en R

| Necesidad | Paquete / función |
|---|---|
| Gráfico estático de calidad | `ggplot2` (`geom_*`, `scale_*`, `facet_*`, `theme()`, `labs()`) |
| Preparar datos para graficar | `dplyr` (`filter`, `mutate`, `group_by`, `summarise`), `tidyr` (`pivot_longer`), `forcats` (`fct_reorder`) |
| Etiquetas de ejes | `scales` (`label_comma`, `label_percent`, `label_dollar`) |
| Colores | `viridis`, `RColorBrewer`, `palette.colors("Okabe-Ito")`, `scale_*_manual` |
| Etiquetas sin superposición | `ggrepel` |
| Combinar gráficos | `patchwork`, `cowplot` |
| Distribuciones por grupo | `ggridges`, `ggpubr`, `ggstatsplot` |
| Correlaciones y pares | `ggcorrplot`, `corrplot`, `GGally::ggpairs` |
| Mapas | `sf` + `geom_sf`, `tmap`, `leaflet`, `rnaturalearth` |
| Interactividad | `plotly` (`ggplotly`), `highcharter`, `echarts4r`, `ggiraph`, `DT` |
| Animación | `gganimate` + `gifski` |
| Redes, árboles, flujos | `visNetwork`, `networkD3`, `ggraph`, `ggalluvial`, `treemapify` |
| Tablas bonitas | `gt`, `kableExtra`, `DT` |
| Construir gráficos sin código | `esquisse` (arrastrar y soltar dentro de RStudio) |
| Reportes reproducibles | `rmarkdown`, `quarto` |
| Dashboards y apps | `flexdashboard`, `shiny`, `bslib`, Quarto dashboards |
| Exportar | `ggsave()`, `htmlwidgets::saveWidget()` |

# Recursos para seguir aprendiendo

- **R for Data Science** (Wickham, Çetinkaya-Rundel & Grolemund, 2.ª ed.): [r4ds.hadley.nz](https://r4ds.hadley.nz) — gratuito, con capítulos de visualización y comunicación.
- **ggplot2: Elegant Graphics for Data Analysis** (Wickham): [ggplot2-book.org](https://ggplot2-book.org) — la referencia definitiva.
- **The R Graph Gallery**: [r-graph-gallery.com](https://r-graph-gallery.com) — cientos de gráficos con su código.
- **Cheatsheets de Posit**: [posit.co/resources/cheatsheets](https://posit.co/resources/cheatsheets/) — ggplot2, dplyr y Shiny en una hoja.
- **TidyTuesday**: [github.com/rfordatascience/tidytuesday](https://github.com/rfordatascience/tidytuesday) — un dataset nuevo cada semana para practicar.
- **Manual de Francisco Charte** (*Análisis exploratorio y visualización de datos con R*) y **curso de Luis Álvarez** — recomendados en la presentación.
- **R-bloggers**, **Stack Overflow** (`[r] [ggplot2]`) y `#rstats` para resolver dudas.

> El camino empieza con un solo paso: abre RStudio, carga un dataset y crea tu primer gráfico. Luego cámbialo, rómpelo y vuélvelo a armar.